# Flow Matching

This notebook is meant as tutorial for flow matching models as part of the paper "A comparison of generative deep learning methods for multivariate angular simulation". We start by importing some modules.

In [ ]:
import h5py
import numpy as np
import pandas as pd

# The models are implemented in the deep learning framework pytorch
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

# We use the flow_matching library for the models. This can be installed with !pip install flow_matching.
from flow_matching.path import GeodesicProbPath
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.solver import RiemannianODESolver
from flow_matching.utils import ModelWrapper
from flow_matching.utils.manifolds import Manifold, Sphere

from tqdm import tqdm


We will apply our model to the sparse gaussian double Pareto example from the paper. We start by reading in the data. We choose the 10 dimensional case with 10 000 samples:

In [ ]:
with h5py.File("data/sparse_gaussian_data_double_pareto.h5", "r") as h5file:
    print(h5file.keys())
    data = pd.DataFrame(h5file["dataset_n_10000_d_10"][:]).transpose().to_numpy()
    

As we are only interested in the angular part we start by projecting the data onto the unit sphere. Also we split it into training and validation set. The validation set is there to determine when to stop the neural network training:

In [ ]:
data_sphere = data / np.linalg.norm(data, axis=-1, keepdims=True)
data_sphere_train = data_sphere[: int(data_sphere.shape[0] * 0.8), :]
data_sphere_val = data_sphere[int(data_sphere.shape[0] * 0.8) :, :]

Next we define the model:

In [ ]:
# Activation class. This provides nonlinearities for our neural network model.
class Swish(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor:
        return torch.sigmoid(x) * x


# Model class. This is what we train.
class MLP(nn.Module):
    def __init__(self, input_dim: int = 2, time_dim: int = 1, hidden_dim: int = 128, num_hidden=3):
        super().__init__()

        self.input_dim = input_dim
        self.time_dim = time_dim
        self.hidden_dim = hidden_dim

        self.input_layer = nn.Linear(input_dim + time_dim, hidden_dim)

        self.main = nn.Sequential(
            *(num_hidden * [Swish(), nn.Linear(hidden_dim, hidden_dim)]), Swish(), nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        sz = x.size()
        x = x.reshape(-1, self.input_dim)
        t = t.reshape(-1, self.time_dim).float()

        t = t.reshape(-1, 1).expand(x.shape[0], 1)
        h = torch.cat([x, t], dim=1)
        h = self.input_layer(h)
        output = self.main(h)

        return output.reshape(*sz)

# This projects the model onto the manifold of choosing, here the unit sphere.
class ProjectToTangent(nn.Module):
    """Projects a vector field onto the tangent plane at the input."""

    def __init__(self, vecfield: nn.Module, manifold: Manifold):
        super().__init__()
        self.vecfield = vecfield
        self.manifold = manifold

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        x = self.manifold.projx(x)
        v = self.vecfield(x, t)
        v = self.manifold.proju(x, v)
        return v


class WrappedModel(ModelWrapper):
    def forward(self, x: torch.Tensor, t: torch.Tensor, **extras):
        return self.model(x=x, t=t)


def wrap(manifold, samples):
    center = torch.cat([torch.zeros_like(samples), torch.ones_like(samples[..., 0:1])], dim=-1)
    samples = torch.cat([samples, torch.zeros_like(samples[..., 0:1])], dim=-1) / 2

    return manifold.expmap(center, samples)

We initialize it for our dataset with 10 dimensions. We choose 4 hidden layers with 128 units each.

In [ ]:
dimension = data_sphere_train.shape[1]
hidden_dim = 128
num_hidden = 4
manifold = Sphere()

# This is the vector field we are training
vf = ProjectToTangent(  # Ensures we can just use Euclidean divergence.
    MLP(  # Vector field in the ambient space.
        input_dim=dimension,
        hidden_dim=hidden_dim,
        num_hidden=num_hidden,
    ),
    manifold=manifold,
)
path = GeodesicProbPath(scheduler=CondOTScheduler(), manifold=manifold)



We check whether a GPU is available and move the model onto there. Also, we convert the training and validation data from numpy to pytorch and also move it onto the GPU if is available. For minibatching we define dataloaders. These progressively output randomly sampled batches of data:

In [ ]:
# Device setup
enable_cuda = True
device = torch.device("cuda" if torch.cuda.is_available() and enable_cuda else "cpu")
vf = vf.to(device)

# Convert data to torch tensors
data_sphere_train = torch.tensor(data_sphere_train).float().to(device)
data_sphere_val = torch.tensor(data_sphere_val).float().to(device)

# DataLoader for training sets
batch_size = 256
train_dataloader = DataLoader(TensorDataset(data_sphere_train), batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(TensorDataset(data_sphere_val), batch_size=batch_size, shuffle=True)

We set a number of training parameters and define the optimizer:

In [ ]:
# Training parameters
max_iter = 5001  
patience = 500
min_delta = 1e-4
best_val_loss = np.inf
counter = 0

# Loss history
loss_hist = np.array([])
loss_hist_val = np.array([])

# Optimizer
optimizer = torch.optim.Adam(vf.parameters(), lr=3e-4)

Finally our training loop. This one iterates over the dataset for `max_iter` iterations (epochs) until early stopping. In each step we iterate through the training dataloader and:

- Generate latent noise using a multivariate normal and wrap it onto the sphere.
- Generate a probability path between the latent noise and the current training data, which we evaluate at a random point $t \in [0,1]$.
- Compute the flow matching loss of the model against this probability path.
- Perform backpropagation.

Finally, after each epoch we evaluate the loss on the validation dataset to determine early stopping:

In [ ]:
# Training loop
for it in tqdm(range(max_iter), desc="Training"):  # Iterate over the maximum number of training iterations
    vf.train()  # Set the model to training mode
    batch_losses = []  # List to store loss values for each batch

    # Loop over mini-batches from the training data
    for batch in train_dataloader:
        x_1 = batch[0]  # Extract the data from the batch
        optimizer.zero_grad()  # Reset the gradients before each batch update

        # Sample x0 from a normal distribution and move it to the appropriate device
        x_0 = torch.randn((x_1.shape[0], dimension - 1)).to(device) # Create latent noise
        x_0 = wrap(manifold, x_0)  # Apply the wrapping function to wrap the latent noise onto the sphere

        # Sample a random time step t between 0 and 1
        t = torch.rand(x_1.shape[0]).to(device)

        # Generate a probability path sample between x_0 and x_1 at time t
        path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)

        # Compute the flow matching L2 loss
        loss = torch.pow(vf(path_sample.x_t, path_sample.t) - path_sample.dx_t, 2).mean()

        # Perform backpropagation and optimizer step only if the loss is finite
        if not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()  # Compute gradients
            optimizer.step()  # Update model parameters
            batch_losses.append(loss.item())  # Store batch loss

    # Compute and store the average loss for this epoch
    epoch_loss = np.mean(batch_losses)
    loss_hist = np.append(loss_hist, epoch_loss)

    # Validation phase (evaluating on validation data without updating model parameters)
    vf.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation for efficiency
        batch_losses_val = []  # List to store validation losses
        for batch in val_dataloader:
            x_1 = batch[0]  # Extract validation data
            x_0 = torch.randn((x_1.shape[0], dimension - 1)).to(device)
            x_0 = wrap(manifold, x_0)  # Wrap x_0 based on the manifold
            t = torch.rand(x_1.shape[0]).to(device)  # Sample random time steps
            path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)  # Generate probability path
            loss = torch.pow(vf(path_sample.x_t, path_sample.t) - path_sample.dx_t, 2).mean()  # Compute loss
            batch_losses_val.append(loss.item())  # Store validation loss
        
        # Compute and store the average validation loss for this epoch
        epoch_val_loss = np.mean(batch_losses_val)
    loss_hist_val = np.append(loss_hist_val, epoch_val_loss)

    # Early stopping mechanism to prevent overfitting
    if epoch_val_loss < best_val_loss - min_delta:
        best_val_loss = epoch_val_loss  # Update the best validation loss
        counter = 0  # Reset early stopping counter
    else:
        counter += 1  # Increment counter if no improvement

    # Stop training if validation loss hasn't improved for 'patience' iterations
    if counter >= patience:
        print(f"Early stopping triggered at iteration {it+1}.")
        break


After the model is trained we can sample from it. For this we need to solve the ODE:

In [ ]:
n_samples = int(1e6) # 1 Million samples

wrapped_vf = WrappedModel(vf)
x_init = torch.randn((n_samples, dimension - 1), dtype=torch.float32, device=device) # Create latent noise
x_init = wrap(manifold, x_init) # Apply the wrapping function to wrap the latent noise onto the sphere

solver = RiemannianODESolver(velocity_model=wrapped_vf, manifold=manifold)  # Create an ODESolver
sol = solver.sample(
    x_init=x_init,
    step_size=0.01,
    method="midpoint",
    return_intermediates=True,
    verbose=False,
) # Solve the ODE associated with the flow matching vector field

samples = sol[-1, :, :].detach().cpu().numpy() # Use only the flow at t = 1

`samples` now contains samples from the trained vector field on the unit sphere. Using a function from `helpers.py` we can transform this into the space of angles:

In [ ]:
R_samples, angles_samples = cartesian_to_polar(samples)